# Stepper Motor Full-Step Control Assignment

In this lab, you will implement a **full-step drive sequence** for a simplified 2-phase stepper motor model.

The notebook includes:
- A pre-built motor simulator
- A visualization of coils and rotor magnets
- An animation to validate your sequence

Color convention:
- **Red** = North pole
- **Blue** = South pole
- Gray = coil off

## Your Task

Implement `build_full_step_sequence(num_steps)` so that it returns a valid **full-step** sequence where both phases are energized on each step.

State format: `(A, B)` where:
- `A` controls top/bottom coils
- `B` controls right/left coils
- each value is in `{ -1, 0, +1 }`

Meaning:
- `+1` means first coil in that phase is North, opposite is South
- `-1` reverses that polarity
- `0` means that phase is off

For this assignment, a full-step state should keep both phases on (`A != 0` and `B != 0`).

In [1]:
%pip install matplotlib

import math
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False


class StepperMotorSimulator:
    """Simple 2-phase stepper motor model for teaching full-step sequencing."""

    def __init__(self):
        # Rotor north pole angle (degrees), 0 degrees points to +x (right).
        self.rotor_angle = 0.0
        self.current_state = (0, 0)

    def reset(self):
        self.rotor_angle = 0.0
        self.current_state = (0, 0)

    def apply_state(self, state):
        # state = (A, B), where A drives top/bottom and B drives right/left.
        A, B = state
        self.current_state = (A, B)

        if A == 0 and B == 0:
            return self.rotor_angle

        # Electromagnetic field direction from combined phase vector.
        target_angle = math.degrees(math.atan2(A, B))
        self.rotor_angle = target_angle
        return self.rotor_angle

    def trace(self, sequence):
        history = [((0, 0), self.rotor_angle)]
        for state in sequence:
            angle = self.apply_state(state)
            history.append((state, angle))
        return history


def pole_colors_for_state(state):
    A, B = state
    off = "#b0b0b0"
    red = "#d62728"
    blue = "#1f77b4"

    # Top / Bottom from phase A
    if A == 1:
        top, bottom = red, blue
    elif A == -1:
        top, bottom = blue, red
    else:
        top, bottom = off, off

    # Right / Left from phase B
    if B == 1:
        right, left = red, blue
    elif B == -1:
        right, left = blue, red
    else:
        right, left = off, off

    return {
        "top": top,
        "right": right,
        "bottom": bottom,
        "left": left,
    }


def draw_motor(ax, state, rotor_angle, title="Stepper Motor State"):
    ax.clear()
    ax.set_aspect("equal")
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.2, 2.2)
    ax.axis("off")

    poles = pole_colors_for_state(state)

    # Stator ring
    ax.add_patch(Circle((0, 0), 1.75, edgecolor="#333333", facecolor="#f7f7f7", lw=2))

    # Coils around stator
    coils = [
        ("top", (0, 1.6), 0.8, 0.35),
        ("right", (1.6, 0), 0.35, 0.8),
        ("bottom", (0, -1.6), 0.8, 0.35),
        ("left", (-1.6, 0), 0.35, 0.8),
    ]

    for name, (cx, cy), w, h in coils:
        color = poles[name]
        rect = Rectangle((cx - w / 2, cy - h / 2), w, h, facecolor=color, edgecolor="#222222", lw=1.5)
        ax.add_patch(rect)
        ax.text(cx, cy, name[0].upper(), ha="center", va="center", color="white", fontsize=10, fontweight="bold")

    # Rotor body
    rotor_radius = 1.0
    ax.add_patch(Circle((0, 0), rotor_radius, edgecolor="#111111", facecolor="#efefef", lw=2))

    # Rotor magnet poles (N/S) shown at opposite ends.
    theta = math.radians(rotor_angle)
    nx, ny = 0.72 * math.cos(theta), 0.72 * math.sin(theta)
    sx, sy = -nx, -ny

    ax.add_patch(Circle((nx, ny), 0.20, color="#d62728", ec="#222222", lw=1))
    ax.add_patch(Circle((sx, sy), 0.20, color="#1f77b4", ec="#222222", lw=1))
    ax.text(nx, ny, "N", ha="center", va="center", color="white", fontsize=10, fontweight="bold")
    ax.text(sx, sy, "S", ha="center", va="center", color="white", fontsize=10, fontweight="bold")

    # Rotor axis marker
    ax.plot([sx, nx], [sy, ny], color="#444444", lw=2)

    A, B = state
    ax.set_title(f"{title}\nState (A, B) = ({A}, {B}) | Rotor angle = {rotor_angle:.0f} deg", fontsize=11)


def animate_sequence(sequence, interval=750):
    sim = StepperMotorSimulator()
    history = sim.trace(sequence)

    fig, ax = plt.subplots(figsize=(6, 6))

    def update(frame_idx):
        state, angle = history[frame_idx]
        draw_motor(ax, state, angle, title="Full-Step Sequence Animation")

    anim = FuncAnimation(fig, update, frames=len(history), interval=interval, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def show_step_slider(sequence):
    if not HAS_WIDGETS:
        print("ipywidgets is not installed; slider view is unavailable.")
        return

    sim = StepperMotorSimulator()
    history = sim.trace(sequence)

    @widgets.interact(step=widgets.IntSlider(min=0, max=len(history) - 1, step=1, value=0, description="Step"))
    def _view(step):
        state, angle = history[step]
        fig, ax = plt.subplots(figsize=(6, 6))
        draw_motor(ax, state, angle, title=f"Manual View (frame {step}/{len(history)-1})")
        plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# STUDENT CELL
# Implement your full-step sequence here.
# NOTE: This cell is pre-populated with a correct solution for testing.
# Remove or replace this before posting the assignment to students.

def build_full_step_sequence(num_steps):
    """
    Return a list of (A, B) states for num_steps steps.

    Rules for this assignment:
    1) Use only full-step two-phase states (A != 0 and B != 0).
    2) Sequence should rotate consistently in one direction.
    3) Repeat pattern as needed to reach num_steps.
    """
    base_pattern = [
        (1, 1),
        (1, -1),
        (-1, -1),
        (-1, 1),
    ]

    return [base_pattern[i % len(base_pattern)] for i in range(num_steps)]

In [ ]:
# Basic validation checks
num_steps = 12
sequence = build_full_step_sequence(num_steps)

assert len(sequence) == num_steps, "Sequence length does not match num_steps."
assert all(len(state) == 2 for state in sequence), "Each state must be a tuple (A, B)."
assert all(a in (-1, 0, 1) and b in (-1, 0, 1) for a, b in sequence), "States must use values in {-1, 0, +1}."
assert all(a != 0 and b != 0 for a, b in sequence), "Full-step two-phase requires A and B both non-zero."

print("Validation passed. First 8 states:")
print(sequence[:8])

In [ ]:
# Animated playback
display(animate_sequence(sequence, interval=700))

# Optional: interactive frame-by-frame slider
show_step_slider(sequence)

## Extension Ideas

1. Reverse direction by reversing the sequence order.
2. Add a half-step mode and compare smoothness.
3. Add dwell time or acceleration profiles between states.